# FakeGuard — AI Fake News Detector
### NeuroLogic '26 Datathon

Run cells **one at a time, top to bottom**.  
Each cell does **one thing only**.  
If a cell fails, fix it and re-run — no need to restart from Cell 1.  

| Cell | Task | Time |
|------|------|------|
| 1 | Fix package conflict (uninstall peft) | 30 sec |
| 2 | Restart kernel | instant |
| 3 | GPU check | instant |
| 4 | Clone GitHub repo | 30 sec |
| 5 | Find dataset | instant |
| 6 | Run preprocessing | 2 min |
| 7 | Run baseline model | 2 min |
| 8 | Train RoBERTa | ~25 min |
| 9 | Evaluate model | 3 min |
| 10 | Generate submission file | 2 min |


## Cell 1 — Fix Package Conflict
`peft` pre-installed on Kaggle conflicts with `transformers`. We uninstall it (we don't use it).

In [ ]:
import subprocess, sys

# Step 1: Remove peft — it conflicts with our transformers version
# We do NOT use peft anywhere, so removing it is safe.
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'peft'])

# Step 2: Install exact versions we need
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.40.0',
    'accelerate==0.29.0',
    'datasets==2.19.0',
    'seaborn',
])

print('Done. Now run Cell 2 to restart the kernel.')

## Cell 2 — Restart Kernel
**Run this cell once.** It restarts the kernel so the package changes take effect.  
After restart, **skip Cell 1 and Cell 2** — start from Cell 3.

In [ ]:
# Restart kernel to reload packages fresh
# After this runs, the notebook will restart automatically.
# Then run from Cell 3 downward.
import IPython
print('Restarting kernel... start from Cell 3 after this.')
IPython.Application.instance().kernel.do_shutdown(restart=True)

## Cell 3 — GPU Check
Confirms GPU is enabled. If it prints `False`, go to Settings → Accelerator → GPU P100.

In [ ]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'GPU available   : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU name        : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('GPU is ready.')
else:
    print('NO GPU FOUND.')
    print('Go to: Notebook Settings → Accelerator → GPU P100 → Save')
    raise RuntimeError('Enable GPU before continuing.')

## Cell 4 — Clone GitHub Repo
Downloads your code from GitHub into Kaggle's working directory.

In [ ]:
import subprocess, os, sys

REPO_URL  = 'https://github.com/ayushtiwari18/neurologic-datathon-fakenews.git'
REPO_ROOT = '/kaggle/working/neurologic-datathon-fakenews'

if not os.path.exists(REPO_ROOT):
    print('Cloning repo...')
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_ROOT])
else:
    print('Repo exists. Pulling latest...')
    subprocess.check_call(['git', '-C', REPO_ROOT, 'pull'])

# Add to Python path so 'from src.train import ...' works
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# Create output folders
for folder in ['data/raw', 'data/processed', 'outputs', 'models/roberta_fakenews']:
    os.makedirs(f'{REPO_ROOT}/{folder}', exist_ok=True)

print(f'Working directory: {os.getcwd()}')
print('Repo is ready.')

## Cell 5 — Find Dataset
Locates `WELFake_Dataset.csv` in `/kaggle/input/`. Make sure you added it via the **Add Data** button on the right panel.

In [ ]:
import glob, shutil
from pathlib import Path

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')

# Search for the CSV in all Kaggle input folders
matches = glob.glob('/kaggle/input/**/WELFake_Dataset.csv', recursive=True)

if matches:
    DATASET_PATH = matches[0]
    print(f'Found dataset: {DATASET_PATH}')
else:
    DATASET_PATH = None
    print('Dataset NOT found in /kaggle/input/')
    print('Fix: Click Add Data (right panel) → find your WELFake dataset → Add')

## Cell 6 — Preprocessing
Splits WELFake into train/val/test CSVs. Skips automatically if already done.

In [ ]:
import pandas as pd
import sys, os
from pathlib import Path

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

train_ready = (PROCESSED_DIR / 'train.csv').exists()
val_ready   = (PROCESSED_DIR / 'val.csv').exists()
test_ready  = (PROCESSED_DIR / 'test.csv').exists()

if train_ready and val_ready and test_ready:
    print('Processed CSVs already exist. Loading them...')
    train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
    val_df   = pd.read_csv(PROCESSED_DIR / 'val.csv')
    test_df  = pd.read_csv(PROCESSED_DIR / 'test.csv')
else:
    import shutil
    if DATASET_PATH is None:
        raise ValueError('Dataset path not set. Run Cell 5 first.')
    raw_dest = Path(f'{REPO_ROOT}/data/raw/WELFake_Dataset.csv')
    shutil.copy2(DATASET_PATH, raw_dest)
    print(f'Copied dataset to {raw_dest}')
    from src.preprocess import run_preprocessing
    train_df, val_df, test_df = run_preprocessing()

print(f'Train : {len(train_df):,} rows')
print(f'Val   : {len(val_df):,} rows')
print(f'Test  : {len(test_df):,} rows')
print('Preprocessing done.')

## Cell 7 — Baseline Model
TF-IDF + Logistic Regression. Takes ~2 minutes. Gives us the comparison score (~94.66%).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
from pathlib import Path

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
val_df   = pd.read_csv(PROCESSED_DIR / 'val.csv')

print('Building TF-IDF features (50k max)...')
vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df['combined'])
X_val   = vectorizer.transform(val_df['combined'])

print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, train_df['label'])

baseline_preds = lr.predict(X_val)
baseline_acc   = accuracy_score(val_df['label'], baseline_preds)
baseline_f1    = f1_score(val_df['label'], baseline_preds, average='weighted')

print(f'Baseline Accuracy : {baseline_acc:.4f}')
print(f'Baseline F1       : {baseline_f1:.4f}')
print(classification_report(val_df['label'], baseline_preds, target_names=['FAKE','REAL']))

## Cell 8 — Train RoBERTa
Fine-tunes `roberta-base` on your training data. Takes ~25 minutes. Model saves every epoch so you won't lose progress.

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from src.train import run_training

print('Starting RoBERTa training...')
print('Expected time: ~25 min on P100 GPU')
print('Checkpoints saved after every epoch.')
print('-' * 50)

trainer, metrics = run_training(
    train_path = str(PROCESSED_DIR / 'train.csv'),
    val_path   = str(PROCESSED_DIR / 'val.csv'),
)

print('-' * 50)
print(f"Val Accuracy : {metrics['final_val_accuracy']:.4f}")
print(f"Val F1       : {metrics['final_val_f1']:.4f}")
print(f"Training time: {metrics['train_runtime_seconds']:.0f} seconds")
print('Training complete.')

## Cell 9 — Evaluate Model
Runs evaluation on the validation set. Saves confusion matrix and error analysis.

In [ ]:
import sys, os
from pathlib import Path
from IPython.display import Image, display

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from src.evaluate import run_evaluation

# baseline_acc comes from Cell 7 — hardcode it here if you restarted
# e.g. baseline_acc = 0.9466
try:
    baseline_acc
except NameError:
    baseline_acc = 0.9466  # fallback: our confirmed local result
    print(f'baseline_acc not found, using confirmed value: {baseline_acc}')

eval_report = run_evaluation(
    val_csv_path      = str(PROCESSED_DIR / 'val.csv'),
    baseline_accuracy = baseline_acc,
)

# Show confusion matrix
cm_path = f'{REPO_ROOT}/outputs/confusion_matrix.png'
if os.path.exists(cm_path):
    display(Image(cm_path))

print(f"RoBERTa Accuracy  : {eval_report['metrics']['accuracy']:.4f}")
print(f"Baseline Accuracy : {baseline_acc:.4f}")
print(f"Improvement       : +{eval_report['metrics']['accuracy'] - baseline_acc:.4f}")

## Cell 10 — Generate Submission File
Runs predictions on the test set and saves `outputs/predictions.csv`. This is your submission file.

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from src.predict import run_predict

predictions_df = run_predict(
    test_csv_path = str(PROCESSED_DIR / 'test.csv'),
)

print(f'Total predictions : {len(predictions_df):,}')
print(f'FAKE predicted    : {(predictions_df["predicted"] == 0).sum():,}')
print(f'REAL predicted    : {(predictions_df["predicted"] == 1).sum():,}')
print()
print('Saved to: outputs/predictions.csv')
print('Download it from the Output panel on the right.')
print('Done!')